-Example

reference :: medium

In [0]:
# pip install llama-index-core llama-index-graph-stores-neo4j
# pip install llama-index-readers-docling graphrag

from llama_index.core import VectorStoreIndex, KnowledgeGraphIndex
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

# ── Build PageIndex (structure-aware, from Docling-parsed docs) ──
from llama_index.readers.docling import DoclingReader
reader    = DoclingReader()
documents = reader.load_data("contracts/")  # preserves sections + page numbers
page_index = VectorStoreIndex.from_documents(documents)
page_engine = page_index.as_query_engine(similarity_top_k=3)

# ── Build Knowledge Graph Index (GraphRAG) ────────────────────
from llama_index.graph_stores.neo4j import Neo4jGraphStore
graph_store  = Neo4jGraphStore(url="bolt://localhost:7687", ...)
kg_index     = KnowledgeGraphIndex.from_documents(
    documents,
    graph_store=graph_store,
    max_triplets_per_chunk=10,   # entities + relationships per chunk
    include_embeddings=True
)
kg_engine = kg_index.as_query_engine(
    include_text=True,
    retriever_mode="keyword",
    response_mode="tree_summarize"  # critical for global queries
)

# ── Router: LLM classifies each query → right engine ─────────
page_tool = QueryEngineTool.from_defaults(
    query_engine=page_engine,
    name="page_index",
    description="Precise fact retrieval from specific documents.
                 Use for: clause lookup, table extraction, page-level facts.
                 NOT for: themes, relationships across documents."
)
graph_tool = QueryEngineTool.from_defaults(
    query_engine=kg_engine,
    name="knowledge_graph",
    description="Relationship reasoning across the full document corpus.
                 Use for: multi-hop queries, themes, entity connections.
                 NOT for: specific page-level facts."
)

router = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[page_tool, graph_tool]
)

# ── Query — router decides which engine to use ───────────────
r1 = router.query("What is the payment term in the Acme contract?")
# → Routes to PageIndex: "Net 30, Page 4, Section 3.1"

r2 = router.query("What liability themes recur across all contracts?")
# → Routes to GraphRAG: synthesises relationship patterns
